In [1]:
import pandas as pd
import kagglehub
import ast
import numpy as np

In [2]:
# Download datasets
path_ml = kagglehub.dataset_download("grouplens/movielens-latest-full")
path_tmdb = kagglehub.dataset_download("rounakbanik/the-movies-dataset")
print("MovieLens path:", path_ml)
print("TMDB path:", path_tmdb)

MovieLens path: /Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1
TMDB path: /Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7


In [3]:
# MovieLens
ratings_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/ratings.csv')
movies_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/movies.csv')
links_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/links.csv')
tags_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/tags.csv')

# TMDB
metadata_tmdb = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/movies_metadata.csv', low_memory=False)
credits_tmdb = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/credits.csv')
keywords_tmdb = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/keywords.csv')

In [4]:
# Clean links
links_ml = links_ml[links_ml['tmdbId'].notnull()]
links_ml['tmdbId'] = links_ml['tmdbId'].astype(int)

# Clean metadata
metadata_tmdb = metadata_tmdb[pd.to_numeric(metadata_tmdb['id'], errors='coerce').notnull()]
metadata_tmdb['id'] = metadata_tmdb['id'].astype(int)
metadata_tmdb['genres'] = metadata_tmdb['genres'].fillna('[]').apply(ast.literal_eval)
metadata_tmdb['main_genre'] = metadata_tmdb['genres'].apply(lambda x: x[0]['name'] if isinstance(x, list) and x else None)

# Parse credits
credits_tmdb['cast'] = credits_tmdb['cast'].apply(ast.literal_eval)
credits_tmdb['crew'] = credits_tmdb['crew'].apply(ast.literal_eval)
credits_tmdb['tmdbId'] = credits_tmdb['id']

def get_director(crew): return next((p['name'] for p in crew if p.get('job') == 'Director'), None)
def get_lead_actor(cast): return cast[0]['name'] if isinstance(cast, list) and cast else None

credits_tmdb['director'] = credits_tmdb['crew'].apply(get_director)
credits_tmdb['lead_actor'] = credits_tmdb['cast'].apply(get_lead_actor)

# Parse keywords
keywords_tmdb['keywords'] = keywords_tmdb['keywords'].fillna('[]').apply(ast.literal_eval)
keywords_tmdb['keywords'] = keywords_tmdb['keywords'].apply(lambda x: [d['name'] for d in x if isinstance(d, dict)])
keywords_tmdb['tmdbId'] = keywords_tmdb['id']

# Aggregate tags
tags_agg = tags_ml.groupby('movieId')['tag'].apply(lambda x: list(set(x))).reset_index()

# Ratings stats
rating_stats = ratings_ml.groupby('movieId')['rating'].agg(['mean', 'min', 'max', 'count']).reset_index()
rating_stats.columns = ['movieId', 'vote_average', 'vote_min', 'vote_max', 'vote_count']

In [5]:
movies_ml_links = pd.merge(movies_ml, links_ml, on='movieId')
metadata_tmdb = metadata_tmdb.rename(columns={'id': 'tmdbId'})
movies_full = pd.merge(movies_ml_links, metadata_tmdb, on='tmdbId', how='inner')

print(movies_full.shape)

movies_full = pd.merge(movies_full, credits_tmdb[['tmdbId', 'director', 'lead_actor']], on='tmdbId', how='left')
movies_full = pd.merge(movies_full, keywords_tmdb[['tmdbId', 'keywords']], on='tmdbId', how='left')
movies_full = pd.merge(movies_full, tags_agg, on='movieId', how='left')
movies_full = pd.merge(movies_full, rating_stats, on='movieId', how='left')
movies_full.shape


(45512, 29)


(46898, 37)

In [6]:
# Runtime binning
bin_edges = list(range(0, 301, 30)) + [np.inf]
labels = [f'{bin_edges[i]}–{bin_edges[i+1]}min' if bin_edges[i+1] != np.inf else f'{bin_edges[i]}min+'
          for i in range(len(bin_edges) - 1)]
movies_full['runtime_bin'] = pd.cut(movies_full['runtime'], bins=bin_edges, labels=labels)

# Year from TMDB release_date
movies_full['release_date_parsed'] = pd.to_datetime(movies_full['release_date'], errors='coerce')
release_year_tmdb = movies_full['release_date_parsed'].dt.year

# Year from MovieLens title
release_year_ml = movies_full['title_x'].str.extract(r'\((\d{4})\)')[0].astype(float)

# Final release year (prefer TMDB, fallback to MovieLens)
movies_full['release_year_tmdb'] = release_year_tmdb
movies_full['release_year_ml'] = release_year_ml
movies_full['release_year'] = release_year_tmdb.combine_first(release_year_ml)

# Earliest year available (in case both exist but differ)
movies_full['release_year_merged'] = movies_full[['release_year_tmdb', 'release_year_ml']].min(axis=1)

# Vote count: max from both sources
movies_full['vote_count'] = movies_full[['vote_count_x', 'vote_count_y']].max(axis=1)

# Vote average from whichever side had more votes
movies_full['vote_average'] = movies_full.apply(
    lambda row: row['vote_average_x'] if row['vote_count_x'] >= row['vote_count_y'] else row['vote_average_y'],
    axis=1
)

# Title: prefer TMDB title
movies_full['title'] = movies_full['title_y'].combine_first(movies_full['title_x'])

In [7]:
# Step 1: Parse genres_x into lists (MovieLens)
movies_full['genres_x_list'] = movies_full['genres_x'].fillna('').apply(lambda x: x.split('|') if isinstance(x, str) else [])

# Step 2: Parse genres_y into lists (TMDB)
movies_full['genres_y_list'] = movies_full['genres_y'].apply(
    lambda x: [d['name'] for d in x if isinstance(d, dict)] if isinstance(x, list) else []
)

# Step 3: Combine both into a unified genre list (set union)
movies_full['genre_list'] = movies_full.apply(
    lambda row: sorted(set(row['genres_x_list']) | set(row['genres_y_list'])),
    axis=1
)

# Step 1: Safely extract main genre from TMDB (if available)
def extract_main_genre(genres_y):
    if isinstance(genres_y, list) and len(genres_y) > 0 and isinstance(genres_y[0], dict):
        return genres_y[0].get('name')  # preserve original TMDB order
    return None

movies_full['main_genre'] = movies_full['genres_y'].apply(extract_main_genre)

movies_full['genres_x_list'] = movies_full['genres_x'].fillna('').apply(lambda x: x.split('|') if isinstance(x, str) else [])

# Use MovieLens main genre if TMDB main genre is missing
movies_full['main_genre'] = movies_full.apply(
    lambda row: row['main_genre'] if pd.notnull(row['main_genre']) else (row['genres_x_list'][0] if row['genres_x_list'] else None),
    axis=1
)
movies_full['genres_x_list'] = movies_full['genres_x'].fillna('').apply(lambda x: x.split('|') if isinstance(x, str) else [])

# Use MovieLens main genre if TMDB main genre is missing
movies_full['main_genre'] = movies_full.apply(
    lambda row: row['main_genre'] if pd.notnull(row['main_genre']) else (row['genres_x_list'][0] if row['genres_x_list'] else None),
    axis=1
)

In [8]:
movies_full['genre_list'][0]

['Adventure', 'Animation', 'Children', 'Comedy', 'Family', 'Fantasy']

In [9]:
from collections import Counter

# Flatten all genre_list entries into one list
all_genres = movies_full['genre_list'].explode()

# Count occurrences using Counter
genre_counts = Counter(all_genres)

# Convert to DataFrame for easy viewing/sorting
genre_count_df = pd.DataFrame(genre_counts.items(), columns=['genre', 'count']).sort_values(by='count', ascending=False)

# Display the result
genre_count_df

,genre,count
7,Drama,23095
3,Comedy,14632
10,Thriller,8965
6,Romance,8238
8,Action,7554
9,Crime,5399
11,Horror,5073
17,Documentary,4415
0,Adventure,4412
13,Mystery,3205


In [10]:
genre_count_df.shape

(26, 2)

In [11]:
movies_full = movies_full.drop(columns=['release_date'])
movies_full = movies_full.rename(columns={'release_date_parsed': 'release_date'})

In [12]:
movies_full = movies_full.drop(columns=[
    'title_x', 'title_y',
    # 'genres_x', 'genres_y',
    'vote_average_x', 'vote_average_y',
    'vote_count_x', 'vote_count_y',
    'release_year_tmdb', 'release_year_ml'
])

In [13]:
[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'name': 'Romance'}]


[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'name': 'Romance'}]

In [14]:
movies_full.iloc[677]

movieId                                                                690
genres_x                                                     Drama|Romance
imdbId                                                              111613
tmdbId                                                              105045
adult                                                                False
belongs_to_collection                                                  NaN
budget                                                                   0
genres_y                 [{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...
homepage                                                               NaN
imdb_id                                                          tt0111613
original_language                                                       de
original_title                                             Das Versprechen
overview                 East-Berlin, 1961, shortly after the erection ...
popularity               

In [15]:
movies_full['release_month'] = movies_full['release_date'].dt.month
movies_full['release_decade'] = (movies_full['release_year'] // 10) * 10

movies_full['popularity_score'] = (
    movies_full['vote_average'] * np.log1p(movies_full['vote_count'])
)

In [16]:
movies_full.columns

Index(['movieId', 'genres_x', 'imdbId', 'tmdbId', 'adult',
       'belongs_to_collection', 'budget', 'genres_y', 'homepage', 'imdb_id',
       'original_language', 'original_title', 'overview', 'popularity',
       'poster_path', 'production_companies', 'production_countries',
       'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'video',
       'main_genre', 'director', 'lead_actor', 'keywords', 'tag', 'vote_min',
       'vote_max', 'runtime_bin', 'release_date', 'release_year',
       'release_year_merged', 'vote_count', 'vote_average', 'title',
       'genres_x_list', 'genres_y_list', 'genre_list', 'release_month',
       'release_decade', 'popularity_score'],
      dtype='object')